In [6]:
!pip install schemdraw sympy pillow google-generativeai openai anthropic scikit-image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 17.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.1/267.1 kB 8.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import random

RANDOM_SEED = 1
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [3]:
import base64

def pil_to_base64(pil_image):
    """Converts a PIL Image to a Base64 string for API transmission."""
    buffered = io.BytesIO()
    pil_image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

Original perturbation code adapted to hold the perturbed image in memory instead of saving directly to a file. No changes to how the code actually runs.

In [7]:
import io
import random
import numpy as np
import skimage
from skimage import transform as sk_transform
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter

def generate_random_displacement(max_displacement):
    return random.uniform(-max_displacement, max_displacement)

def simulate_photocopy_scan_cycles_memory(input_image, cycles,
                                          contrast_range=(1.1, 1.4),
                                          brightness_range=(0.95, 1.05),
                                          blur_radius_range=(0.3, 0.8),
                                          noise=True,
                                          noise_intensity_range=(0.00025, 0.0005),
                                          max_rotation_angle_deg=1,
                                          grayscale_all_cycles=False,
                                          jpeg_compression_quality_range=(50, 90)):
    """In-memory version of the photocopy degradation function."""
    current_image_pil = input_image.copy()

    if current_image_pil.mode == 'RGBA':
        background = Image.new('RGB', current_image_pil.size, (255, 255, 255))
        background.paste(current_image_pil, mask=current_image_pil.split()[3])
        current_image_pil = background
    elif current_image_pil.mode == 'P':
        current_image_pil = current_image_pil.convert('RGB')

    original_width, original_height = current_image_pil.size

    for i in range(cycles):
        if grayscale_all_cycles and current_image_pil.mode != 'L':
            current_image_pil = current_image_pil.convert('L')
        if not grayscale_all_cycles and current_image_pil.mode != 'RGB':
             current_image_pil = current_image_pil.convert('RGB')

        # 1. Contrast & Brightness
        contrast_factor = random.uniform(contrast_range[0], contrast_range[1])
        current_image_pil = ImageEnhance.Contrast(current_image_pil).enhance(contrast_factor)

        brightness_factor = random.uniform(brightness_range[0], brightness_range[1])
        current_image_pil = ImageEnhance.Brightness(current_image_pil).enhance(brightness_factor)

        # 2. Gaussian Blur
        blur_radius = random.uniform(blur_radius_range[0], blur_radius_range[1])
        if blur_radius > 0:
            current_image_pil = current_image_pil.filter(ImageFilter.GaussianBlur(radius=blur_radius))

        # 3. Rotation
        angle = random.uniform(-max_rotation_angle_deg, max_rotation_angle_deg)
        if angle != 0:
            fill_color = 'white' if current_image_pil.mode == 'L' else (255, 255, 255)
            rotated_image = current_image_pil.rotate(angle, resample=Image.BICUBIC, expand=True, fillcolor=fill_color)
            w_rot, h_rot = rotated_image.size
            x_offset = (w_rot - original_width) / 2
            y_offset = (h_rot - original_height) / 2
            current_image_pil = rotated_image.crop((x_offset, y_offset, x_offset + original_width, y_offset + original_height))

        # 4. Noise
        if noise:
            noise_intensity = random.uniform(noise_intensity_range[0], noise_intensity_range[1])
            num_noise_dots = int(original_width * original_height * noise_intensity)
            draw = ImageDraw.Draw(current_image_pil)
            for _ in range(num_noise_dots):
                x, y = random.randint(0, original_width - 1), random.randint(0, original_height - 1)
                noise_shade = random.randint(0, 100)
                dot_color = noise_shade if current_image_pil.mode == 'L' else (noise_shade, noise_shade, noise_shade)
                draw.point((x, y), fill=dot_color)
            del draw

        # 5. Simulate JPEG Compression entirely in memory!
        if cycles > 0:
            quality = random.randint(jpeg_compression_quality_range[0], jpeg_compression_quality_range[1])
            img_buffer = io.BytesIO()
            # Save to buffer instead of disk
            current_image_pil.save(img_buffer, "JPEG", quality=quality)
            img_buffer.seek(0)
            # Reload from buffer
            current_image_pil = Image.open(img_buffer)

            if grayscale_all_cycles and current_image_pil.mode != 'L':
                current_image_pil = current_image_pil.convert('L')
            elif not grayscale_all_cycles and current_image_pil.mode not in ['RGB', 'L']:
                current_image_pil = current_image_pil.convert('RGB')

    return current_image_pil

def apply_perturbations(input_pil_image,
                        grid_granularity=(5, 7),
                        max_displacement_ratio=0.05,
                        affine=True, noise=True, lines=False, photocopy=False, photocopy_cycles=5):
    try:
        image_sk = np.array(input_pil_image)
        if image_sk.shape[-1] == 4:
            image_sk = skimage.color.rgba2rgb(image_sk)

        rows, cols = image_sk.shape[0], image_sk.shape[1]

        # Warping Logic
        src_cols = np.linspace(0, cols, grid_granularity[1])
        src_rows = np.linspace(0, rows, grid_granularity[0])
        src_rows, src_cols = np.meshgrid(src_rows, src_cols)
        src = np.dstack([src_cols.flat, src_rows.flat])[0]

        max_disp = min(rows, cols) * max_displacement_ratio
        dst = np.zeros_like(src)
        for i in range(len(src)):
            disp_x = generate_random_displacement(max_disp)
            disp_y = generate_random_displacement(max_disp)
            dst_x = np.clip(src[i, 0] + disp_x, 0, cols)
            dst_y = np.clip(src[i, 1] + disp_y, 0, rows)
            dst[i] = (dst_x, dst_y)

        tform = sk_transform.PiecewiseAffineTransform()
        tform.estimate(src, dst)

        if affine:
            warped_sk = sk_transform.warp(image_sk, tform.inverse, output_shape=(rows, cols), mode='edge')
            warped_pil = Image.fromarray((warped_sk * 255).astype(np.uint8))
        else:
            warped_pil = input_pil_image.copy()

        draw = ImageDraw.Draw(warped_pil)
        width, height = warped_pil.size

        # Simple Noise
        if noise:
            num_noise_dots = int(width * height * random.uniform(0.01, 0.02))
            for _ in range(num_noise_dots):
                x, y = random.randint(0, width - 1), random.randint(0, height - 1)
                noise_shade = random.randint(50, 200)
                noise_color = (noise_shade, noise_shade, noise_shade)
                if warped_pil.mode == 'RGBA':
                    noise_color += (random.randint(100, 255),)
                draw.point((x, y), fill=noise_color)

        # Lines
        if lines:
            num_lines = random.randint(3, 7)
            for _ in range(num_lines):
                start_x, start_y = random.randint(0, width - 1), random.randint(0, height - 1)
                end_x, end_y = random.randint(0, width - 1), random.randint(0, height - 1)
                line_shade = random.randint(70, 180)
                line_color = (line_shade, line_shade, line_shade)
                if warped_pil.mode == 'RGBA':
                    line_color += (random.randint(100,200),)
                draw.line([(start_x, start_y), (end_x, end_y)], fill=line_color, width=random.randint(1, 2))

        # Photocopy Integration
        if photocopy:
            warped_pil = simulate_photocopy_scan_cycles_memory(warped_pil, photocopy_cycles)

        return warped_pil

    except Exception as e:
        print(f"Error during perturbation: {e}")
        import traceback
        traceback.print_exc()
        return None

Circuit dataset generator

In [8]:
import itertools
import random
import hashlib
from sympy import symbols, And, Or, Xor, Nand, Nor
from sympy.logic.boolalg import truth_table

GATE_TYPES = [
    (And, "AND"),
    (Or, "OR"),
    (Xor, "XOR"),
    (Nand, "NAND"),
    (Nor, "NOR")
]

def generate_circuit_datasets(export_variants=False):
    A, B, C, D = symbols('A B C D')
    all_structures = list(itertools.product(GATE_TYPES, repeat=3))

    # Use a fixed seed for repeatable dataset generation
    random.seed(RANDOM_SEED)
    random.shuffle(all_structures)

    # Dictionary to group circuits by their exact logical truth table
    # Format: { "truth_signature": [ circuit_1, circuit_2, ... ] }
    isomorphic_groups = {}

    for (gate1_sym, name1), (gate2_sym, name2), (gate3_sym, name3) in all_structures:
        # Build expression
        expr_top = gate1_sym(A, B)
        expr_bot = gate2_sym(C, D)
        ground_truth_expr = gate3_sym(expr_top, expr_bot)

        # Extract truth table signature
        table = list(truth_table(ground_truth_expr, [A, B, C, D]))
        output_signature = tuple(result[1] for result in table)

        # Create a unique ID for the logical family (makes tracking variants easy)
        family_id = hashlib.md5(str(output_signature).encode()).hexdigest()[:8]

        circuit_data = {
            "family_id": family_id,
            "gate1": name1,
            "gate2": name2,
            "gate3": name3,
            "expression": str(ground_truth_expr),
            "signature": output_signature
        }

        if output_signature not in isomorphic_groups:
            isomorphic_groups[output_signature] = []

        isomorphic_groups[output_signature].append(circuit_data)

    # Now, split the groups into the Primary Dataset and the Quarantined Variants
    primary_dataset = []
    variant_dataset = {} # Map family_id to its list of variants

    for signature, circuit_list in isomorphic_groups.items():
        # The first circuit discovered for this logic becomes the Primary representative
        primary_circuit = circuit_list[0]
        primary_dataset.append(primary_circuit)

        # Any subsequent circuits with the same logic are stored as variants
        if len(circuit_list) > 1:
            variant_dataset[primary_circuit["family_id"]] = circuit_list[1:]

    print(f"Generated {len(primary_dataset)} logically unique base problems.")
    print(f"Quarantined {sum(len(v) for v in variant_dataset.values())} structural variants.")

    if export_variants:
        return primary_dataset, variant_dataset
    else:
        return primary_dataset

if __name__ == "__main__":
    # Default behavior: returns only the logically unique set
    primary_data = generate_circuit_datasets()

    # When you are ready to test spatial robustness:
    # primary_data, variant_data = generate_circuit_datasets(export_variants=True)

Generated 85 logically unique base problems.
Quarantined 40 structural variants.


Gemini configuration code.


In [ ]:
import os
import re
import time
import google.generativeai as genai
from sympy import sympify
from sympy.logic.boolalg import Equivalent


genai.configure(api_key="YOUR_KEY_HERE")  # Replace with your actual API key

def evaluate_with_gemini(pil_image, ground_truth_expr):
    """
    Sends the perturbed image to Gemini, extracts the logical answer,
    and mathematically verifies it against the ground truth.
    """
    # 1. Define the deterministic model settings
    # We use temperature=0.0 to minimize stochastic "creativity" in math problems
    generation_config = {
        "temperature": 0.0,
        "top_p": 1.0,
    }

    # 2. Initialize the Pro model
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        generation_config=generation_config
    )

    # 3. The Standardized System Prompt
    # We force the model to use a specific syntax so we can easily parse it
    prompt = """
    You are an expert computer engineering professor. 
    Analyze the digital logic circuit in the provided image.
    Write out the final boolean expression for the output F in terms of the inputs.
    
    You MUST provide your final answer on a new line in this exact format:
    FINAL ANSWER: <expression>
    
    CRITICAL SYNTAX RULES for the final expression:
    1. Use uppercase letters for variables (A, B, C, D).
    2. You MUST use the tilde (~) for NOT. Do NOT use apostrophes or primes (A').
    3. You MUST use the ampersand (&) for AND. Do NOT write variables next to each other (AB).
    4. You MUST use the pipe (|) for OR. Do NOT use the plus sign (+).
    5. Do NOT use the caret symbol (^) for XOR. Expand XOR into standard SOP form.
    
    EXAMPLE OF BAD OUTPUT: A'B + CD'
    EXAMPLE OF GOOD OUTPUT: ~A & B | C & ~D
    """

    try:
        # 4. The API Call (Passing the PIL image directly!)
        response = model.generate_content([prompt, pil_image])
        model_text = response.text

        # 5. Extract the answer using Regex
        match = re.search(r"FINAL ANSWER:\s*(.+)", model_text, re.IGNORECASE)

        if not match:
            return {
                "success": False,
                "error": "Formatting Failure: Model did not use 'FINAL ANSWER:' tag.",
                "raw_output": model_text
            }

        model_expr_str = match.group(1).strip()

        # 6. Mathematical Verification via SymPy
        try:
            # Parse the model's text string using the same logic rules
            model_sym = parse_expr(model_expr_str, transformations=LOGIC_TRANSFORMATIONS)

            # Force SymPy to evaluate the equivalence and cast to a native Python boolean
            is_correct = bool(Equivalent(ground_truth_expr, model_sym))

            return {
                "success": True,
                "is_correct": is_correct,
                "model_expression": model_expr_str,
                "ground_truth": str(ground_truth_expr),
                "raw_output": model_text
            }

        except Exception as math_err:
            return {
                "success": False,
                "error": f"Parse Error: Could not convert model output to SymPy logic. ({math_err})",
                "raw_output": model_text
            }

    except Exception as api_err:
        return {
            "success": False,
            "error": f"API Error: {api_err}"
        }

Helper code to see available Gemini models.

Use this if model not supported errors in the main driver.

In [ ]:
# @title
# import os
# import google.generativeai as genai

# # Ensure your API key is configured
# genai.configure(api_key="YOUR KEY HERE")

# print("Supported Vision & Text Models:")
# print("-" * 30)

# # Loop through and list models that support standard text/image generation
# for m in genai.list_models():
#     if 'generateContent' in m.supported_generation_methods:
#         print(m.name)

GPT Evaluate Function

In [ ]:
import os
import re
from openai import OpenAI
from sympy.parsing.sympy_parser import parse_expr
from sympy.logic.boolalg import Equivalent

# Initialize the OpenAI client
openai_client = OpenAI(api_key="YOUR_KEY_HERE")  # Replace with your actual API key

def evaluate_with_openai(pil_image, ground_truth_sym):
    base64_image = pil_to_base64(pil_image)

    prompt = """
    You are an expert computer engineering professor. 
    Analyze the digital logic circuit in the provided image.
    Write out the final boolean expression for the output F in terms of the inputs.
    
    You MUST provide your final answer on a new line in this exact format:
    FINAL ANSWER: <expression>
    
    CRITICAL SYNTAX RULES for the final expression:
    1. Use uppercase letters for variables (A, B, C, D).
    2. You MUST use the tilde (~) for NOT. Do NOT use apostrophes or primes (A').
    3. You MUST use the ampersand (&) for AND. Do NOT write variables next to each other (AB).
    4. You MUST use the pipe (|) for OR. Do NOT use the plus sign (+).
    5. Do NOT use the caret symbol (^) for XOR. Expand XOR into standard SOP form.
    
    EXAMPLE OF BAD OUTPUT: A'B + CD'
    EXAMPLE OF GOOD OUTPUT: ~A & B | C & ~D
    """

    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0.0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ]
        )

        model_text = response.choices[0].message.content
        match = re.search(r"FINAL ANSWER:\s*(.+)", model_text, re.IGNORECASE)

        if not match:
            return {"success": False, "error": "Formatting Failure.", "raw_output": model_text}

        model_expr_str = match.group(1).strip()

        try:
            # Use the strict logic transformations defined earlier
            model_sym = parse_expr(model_expr_str, transformations=LOGIC_TRANSFORMATIONS)
            is_correct = bool(Equivalent(ground_truth_sym, model_sym))

            return {
                "success": True,
                "is_correct": is_correct,
                "model_expression": model_expr_str,
                "ground_truth": str(ground_truth_sym),
                "raw_output": model_text
            }
        except Exception as math_err:
            return {"success": False, "error": f"Parse Error: {math_err}", "raw_output": model_text}

    except Exception as api_err:
        # Note: OpenAI uses different rate limit error codes, but the string cast will still catch "429"
        return {"success": False, "error": f"API Error: {api_err}"}

Claude Evaluate Function


In [ ]:
import os
import re
from anthropic import Anthropic
from sympy.parsing.sympy_parser import parse_expr
from sympy.logic.boolalg import Equivalent

# Initialize the Anthropic client
anthropic_client = Anthropic(api_key="YOUR_KEY_HERE")  # Replace with your actual API key

def evaluate_with_anthropic(pil_image, ground_truth_sym):
    base64_image = pil_to_base64(pil_image)

    prompt = """
    You are an expert computer engineering professor. 
    Analyze the digital logic circuit in the provided image.
    Write out the final boolean expression for the output F in terms of the inputs.
    
    You MUST provide your final answer on a new line in this exact format:
    FINAL ANSWER: <expression>
    
    CRITICAL SYNTAX RULES for the final expression:
    1. Use uppercase letters for variables (A, B, C, D).
    2. You MUST use the tilde (~) for NOT. Do NOT use apostrophes or primes (A').
    3. You MUST use the ampersand (&) for AND. Do NOT write variables next to each other (AB).
    4. You MUST use the pipe (|) for OR. Do NOT use the plus sign (+).
    5. Do NOT use the caret symbol (^) for XOR. Expand XOR into standard SOP form.
    
    EXAMPLE OF BAD OUTPUT: A'B + CD'
    EXAMPLE OF GOOD OUTPUT: ~A & B | C & ~D
    """

    try:
        response = anthropic_client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=300,
            temperature=0.0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/png",
                                "data": base64_image
                            }
                        },
                        {"type": "text", "text": prompt}
                    ]
                }
            ]
        )

        model_text = response.content[0].text
        match = re.search(r"FINAL ANSWER:\s*(.+)", model_text, re.IGNORECASE)

        if not match:
            return {"success": False, "error": "Formatting Failure.", "raw_output": model_text}

        model_expr_str = match.group(1).strip()

        try:
            model_sym = parse_expr(model_expr_str, transformations=LOGIC_TRANSFORMATIONS)
            is_correct = bool(Equivalent(ground_truth_sym, model_sym))

            return {
                "success": True,
                "is_correct": is_correct,
                "model_expression": model_expr_str,
                "ground_truth": str(ground_truth_sym),
                "raw_output": model_text
            }
        except Exception as math_err:
            return {"success": False, "error": f"Parse Error: {math_err}", "raw_output": model_text}

    except Exception as api_err:
        return {"success": False, "error": f"API Error: {api_err}"}

Main Execution Block:

Generates logical circuit diagrams then passes them to the Gemini API.

In [ ]:
import io
import json
import time
import datetime
import os
from pathlib import Path
import schemdraw
import schemdraw.logic as logic
from sympy import sympify
from PIL import Image




from sympy.parsing.sympy_parser import parse_expr, standard_transformations, convert_xor

# Define a logic-parsing environment that prevents ^ from becoming Power (**)
LOGIC_TRANSFORMATIONS = tuple(t for t in standard_transformations if t != convert_xor)


# ---------------------------------------------------------
# 1. Modular Generator Function
# ---------------------------------------------------------
# Map string names from the dataset back to schemdraw objects
GATE_MAP = {
    "AND": logic.And,
    "OR": logic.Or,
    "XOR": logic.Xor,
    "NAND": logic.Nand,
    "NOR": logic.Nor
}

def generate_circuit_image(circuit_data):
    """Generates a clean PIL image dynamically from dataset specifications."""
    Gate1_Draw = GATE_MAP[circuit_data['gate1']]
    Gate2_Draw = GATE_MAP[circuit_data['gate2']]
    Gate3_Draw = GATE_MAP[circuit_data['gate3']]

    with schemdraw.Drawing(show=False) as d:
        d.config(fontsize=14, unit=2.5)

        # --- Top Branch ---
        in_A = d.add(logic.Dot(lftlabel='A'))
        d.add(logic.Line(d='right', l=1))
        gate1 = d.add(Gate1_Draw(d='right', anchor='in1'))

        in_B = d.add(logic.Dot(xy=[in_A.start[0], gate1.in2[1]], lftlabel='B'))
        d.add(logic.Line(d='right', tox=gate1.in2[0]))

        # --- Bottom Branch ---
        in_C = d.add(logic.Dot(xy=[in_A.start[0], gate1.in2[1] - 2], lftlabel='C'))
        d.add(logic.Line(d='right', l=1))
        gate2 = d.add(Gate2_Draw(d='right', anchor='in1'))

        in_D = d.add(logic.Dot(xy=[in_A.start[0], gate2.in2[1]], lftlabel='D'))
        d.add(logic.Line(d='right', tox=gate2.in2[0]))

        # --- Output Gate ---
        x_out = gate1.out[0] + 2
        midpoint_y = (gate1.out[1] + gate2.out[1]) / 2
        gate3 = d.add(Gate3_Draw(d='right', xy=[x_out, midpoint_y + 0.5], anchor='in1'))

        # Connect Branches
        d.add(logic.Wire('-|').at(gate1.out).to(gate3.in1))
        d.add(logic.Wire('-|').at(gate2.out).to(gate3.in2))

        # Final Output Line
        d.add(logic.Line(xy=gate3.out, d='right', l=1, rgtlabel='F'))

        # Use get_imagedata to avoid the save format TypeError
        image_bytes = d.get_imagedata('png')
        img_buffer = io.BytesIO(image_bytes)

    return Image.open(img_buffer)




import os
import re
import io
import base64
from sympy.parsing.sympy_parser import parse_expr
from sympy.logic.boolalg import simplify_logic, Equivalent
import google.generativeai as genai
from openai import OpenAI
from anthropic import Anthropic

# --- API CLIENT INITIALIZATION ---
genai.configure(api_key="YOUR_KEY_HERE")  # Replace with your actual API key
openai_client = OpenAI(api_key="YOUR_KEY_HERE")
anthropic_client = Anthropic(api_key="YOUR_KEY_HERE")

def pil_to_base64(pil_image):
    """Converts a PIL Image to a Base64 string for API transmission."""
    buffered = io.BytesIO()
    pil_image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

# --- THE STANDARDIZED CIRCUIT PROMPT ---
CIRCUIT_PROMPT = """
You are an expert computer engineering professor. 
Analyze the digital logic circuit in the provided image. 
Write out the final boolean expression for the output F in terms of the inputs.

You MUST provide your final answer on a new line in this exact format:
FINAL ANSWER: <expression>

CRITICAL SYNTAX RULES for the final expression:
1. Use uppercase letters for variables (A, B, C, D).
2. You MUST use the tilde (~) for NOT. Do NOT use apostrophes or primes (A').
3. You MUST use the ampersand (&) for AND. Do NOT write variables next to each other (AB).
4. You MUST use the pipe (|) for OR. Do NOT use the plus sign (+).
5. Do NOT use the caret symbol (^) for XOR. Expand XOR into standard SOP form.

EXAMPLE OF BAD OUTPUT: A'B + CD'
EXAMPLE OF GOOD OUTPUT: ~A & B | C & ~D
"""

# ==========================================
# 1. GEMINI EVALUATOR
# ==========================================
def evaluate_with_gemini(pil_image, ground_truth_sym):
    # Initialize model locally for thread safety if needed
    model = genai.GenerativeModel(
        model_name="gemini-2.5-pro",
        generation_config={"temperature": 0.0, "top_p": 1.0}
    )
    
    try:
        response = model.generate_content([CIRCUIT_PROMPT, pil_image])
        model_text = response.text
        return parse_and_verify_logic(model_text, ground_truth_sym)
    except Exception as api_err:
        return {"success": False, "error": f"API Error: {api_err}"}

# ==========================================
# 2. OPENAI EVALUATOR
# ==========================================
def evaluate_with_openai(pil_image, ground_truth_sym):
    base64_image = pil_to_base64(pil_image)
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0.0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": CIRCUIT_PROMPT},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}}
                    ]
                }
            ]
        )
        model_text = response.choices[0].message.content
        return parse_and_verify_logic(model_text, ground_truth_sym)
    except Exception as api_err:
        return {"success": False, "error": f"API Error: {api_err}"}

# ==========================================
# 3. ANTHROPIC EVALUATOR
# ==========================================
def evaluate_with_anthropic(pil_image, ground_truth_sym):
    base64_image = pil_to_base64(pil_image)
    try:
        response = anthropic_client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=1024,
            temperature=0.0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": base64_image}},
                        {"type": "text", "text": CIRCUIT_PROMPT}
                    ]
                }
            ]
        )
        model_text = response.content[0].text
        return parse_and_verify_logic(model_text, ground_truth_sym)
    except Exception as api_err:
        return {"success": False, "error": f"API Error: {api_err}"}

# ==========================================
# 4. THE STRICT MATH PARSER (SHARED)
# ==========================================
def parse_and_verify_logic(model_text, ground_truth_sym):
    """Handles the regex extraction, string cleaning, and SymPy math evaluation."""
    match = re.search(r"FINAL ANSWER:\s*(.+)", model_text, re.IGNORECASE)
    
    if not match:
        return {"success": False, "error": "Formatting Failure: No 'FINAL ANSWER:' tag.", "raw_output": model_text}
        
    model_expr_str = match.group(1).strip()
    
    # --- THE STRING SAFETY NET ---
    model_expr_str = model_expr_str.replace("+", "|") # Fix ORs
    model_expr_str = re.sub(r'([a-zA-Z])\'', r'~\1', model_expr_str) # Fix Primes
    model_expr_str = re.sub(r'([A-Z])\s*(?=[A-Z~])', r'\1 & ', model_expr_str) # Fix Implicit ANDs
    # -----------------------------

    try:
        # Parse using the strict logic transformations defined at the top of your script
        model_sym = parse_expr(model_expr_str, transformations=LOGIC_TRANSFORMATIONS)
        
        # STRICT EQUIVALENCE: Forces SymPy to evaluate the math down to a literal Python True/False
        is_correct = (simplify_logic(Equivalent(ground_truth_sym, model_sym)) == True)
        
        return {
            "success": True,
            "is_correct": is_correct,
            "model_expression": model_expr_str,
            "ground_truth": str(ground_truth_sym),
            "raw_output": model_text
        }
    except Exception as math_err:
        return {"success": False, "error": f"Parse Error: {math_err}", "raw_output": model_text}



# ---------------------------------------------------------
# 2. Main Evaluation Loop
# ---------------------------------------------------------
def run_automated_benchmark(dataset,
                            perturb_config,
                            experiment_name="run_01",
                            sample_size=5,
                            num_passes=3, # <-- New parameter for multi-pass evaluation
                            save_images=True,
                            base_output_dir="benchmark_data"):

    exp_dir = Path(base_output_dir) / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    output_file = exp_dir / "results.json"

    if save_images:
        clean_dir = exp_dir / "clean"
        perturb_dir = exp_dir / "perturbed"
        clean_dir.mkdir(exist_ok=True)
        perturb_dir.mkdir(exist_ok=True)

    if output_file.exists():
        with open(output_file, 'r') as f:
            results_log = json.load(f)
    else:
        results_log = []
        with open(output_file, 'w') as f:
            json.dump([], f)

    print(f"Starting benchmark '{experiment_name}'. {num_passes} passes per image.")

    for i, data in enumerate(dataset[:sample_size]):
        family_id = data['family_id']
        print(f"\n[{i+1}/{sample_size}] Testing Logic Family: {family_id}...")

        try:
            # --- 1. GENERATE ONCE ---
            clean_pil = generate_circuit_image(data)

            # --- 2. PERTURB ONCE (Locking the visual state) ---
            adversarial_pil = apply_perturbations(clean_pil, **perturb_config)

            if not adversarial_pil:
                raise ValueError("Perturbation function returned None.")

            if save_images:
                clean_pil.save(clean_dir / f"{family_id}_clean.png")
                adversarial_pil.save(perturb_dir / f"{family_id}_perturbed.png")

            ground_truth_sym = parse_expr(data['expression'], transformations=LOGIC_TRANSFORMATIONS)

            # --- 3. EVALUATE N TIMES WITH SMART RETRY ---
            for pass_idx in range(1, num_passes + 1):
                print(f"  -> Pass {pass_idx}/{num_passes} for {family_id}...")

                max_retries = 3
                retry_count = 0
                evaluation = None

                while retry_count < max_retries:
                    # We pass the EXACT SAME image to the API
                    evaluation = evaluate_with_gemini(adversarial_pil, ground_truth_sym)
                    #evaluation = evaluate_with_anthropic(adversarial_pil, ground_truth_sym)
                    #evaluation = evaluate_with_openai(adversarial_pil, ground_truth_sym)


                    # Check if it failed specifically due to a 429 Rate Limit
                    if not evaluation.get("success") and "429" in str(evaluation.get("error", "")):
                        error_text = str(evaluation.get("error", ""))

                        # Extract the exact wait time requested by Google
                        match = re.search(r"Please retry in ([0-9.]+)s", error_text)
                        if match:
                            wait_time = float(match.group(1)) + 5.0  # 5-second safety buffer
                        else:
                            wait_time = 60.0  # Fallback if regex misses

                        print(f"     [Rate Limit] Sleeping for {wait_time:.1f}s (Attempt {retry_count + 1}/{max_retries})...")
                        time.sleep(wait_time)
                        retry_count += 1
                        continue # Loop back and try the API call again

                    # If it's a success, or a DIFFERENT kind of error (like a math parse error), break the retry loop
                    break

                # If we exhausted retries and it's still a 429, mark it as a hard error
                if not evaluation.get("success") and "429" in str(evaluation.get("error", "")):
                     evaluation["error"] = "CRITICAL: Exhausted all rate limit retries."

                # --- LOGGING ---
                log_entry = {
                    "timestamp": datetime.datetime.now().isoformat(),
                    "experiment_name": experiment_name,
                    "family_id": family_id,
                    "pass_index": pass_idx,
                    "is_variant": data.get("is_variant", False),
                    "variant_note": data.get("variant_note", ""),
                    "gate_structure": f"{data['gate1']}_{data['gate2']}_{data['gate3']}",
                    "ground_truth_logic": str(ground_truth_sym),
                    "perturbation_config": perturb_config,
                    "api_success": evaluation.get("success", False),
                    "is_correct": evaluation.get("is_correct", None),
                    "model_parsed_logic": evaluation.get("model_expression", None),
                    "raw_model_output": evaluation.get("raw_output", ""),
                    "error_message": evaluation.get("error", None),
                    "requires_manual_review": not evaluation.get("success", False)
                }

                results_log.append(log_entry)

                with open(output_file, 'w') as f:
                    json.dump(results_log, f, indent=4)

                if log_entry["api_success"]:
                    status = "PASS" if log_entry["is_correct"] else "FAIL"
                    print(f"     Result: {status}")
                else:
                    print(f"     Result: FLAG FOR REVIEW ({log_entry['error_message']})")

                # Standard sleep between successful passes
                time.sleep(15)

        except Exception as e:
            print(f"  -> CRITICAL EXCEPTION on {family_id}: {e}")
            log_entry = {
                "timestamp": datetime.datetime.now().isoformat(),
                "experiment_name": experiment_name,
                "family_id": family_id,
                "pass_index": "N/A",
                "error_message": f"SYSTEM EXCEPTION: {str(e)}",
                "requires_manual_review": True
            }
            results_log.append(log_entry)
            with open(output_file, 'w') as f:
                json.dump(results_log, f, indent=4)
            time.sleep(4)

    print(f"\nBenchmark complete. Data safely stored in {exp_dir}")
    return results_log

if __name__ == "__main__":
    # 1. Generate the data and request the variants
    primary_data, variant_data = generate_circuit_datasets(export_variants=True)

    paired_dataset = []
    target_pairs = 40
    pairs_found = 0

    # 2. Build the paired evaluation list
    for primary_circuit in primary_data:
        family_id = primary_circuit['family_id']

        # Check if this specific logic family has at least one structural variant
        if family_id in variant_data and len(variant_data[family_id]) > 0:

            # Tag the primary circuit
            primary_circuit['is_variant'] = False
            primary_circuit['variant_note'] = "Primary"

            # Grab the first available structural variant and tag it
            variant_circuit = variant_data[family_id][0]
            variant_circuit['is_variant'] = True
            variant_circuit['variant_note'] = "Isomorphic Companion"

            # Append them back-to-back so they are evaluated sequentially
            paired_dataset.append(primary_circuit)
            paired_dataset.append(variant_circuit)

            pairs_found += 1
            if pairs_found >= target_pairs:
                break

    print(f"Successfully built {pairs_found} isomorphic pairs ({len(paired_dataset)} total evaluations).")

    # 3. Define the Full Perturbation Suite
    my_perturb_config = {
        "photocopy": True,
        "photocopy_cycles": 3,
        "noise": True,
        "affine": True,
        "lines": True
    }


    clean_config = {
    "photocopy": False,
    "photocopy_cycles": 3,
    "noise": False,
    "affine": False,
    "lines": False
}


    # 4. Execute the Benchmark
    # Note: sample_size is set to the full length of the paired list
    run_automated_benchmark(
        dataset=paired_dataset,
        perturb_config=my_perturb_config,
        #perturb_config=clean_config,
        #experiment_name="paired_full_suite_test_claudeHaiku45",
        experiment_name="ld_gemini25pro_perturbed",
        sample_size=len(paired_dataset),
        num_passes=3,
        save_images=True
    )

Generated 85 logically unique base problems.
Quarantined 40 structural variants.
Successfully built 40 isomorphic pairs (80 total evaluations).
Starting benchmark 'ld_gemini25pro_perturbed'. 3 passes per image.

[1/80] Testing Logic Family: bfab0d3c...
  -> Pass 1/3 for bfab0d3c...


/tmp/ipykernel_662055/655588362.py:116: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PiecewiseAffineTransform.from_estimate` class constructor instead.
  tform.estimate(src, dst)


     Result: PASS
  -> Pass 2/3 for bfab0d3c...
     Result: PASS
  -> Pass 3/3 for bfab0d3c...
     Result: PASS

[2/80] Testing Logic Family: bfab0d3c...
  -> Pass 1/3 for bfab0d3c...
     Result: PASS
  -> Pass 2/3 for bfab0d3c...
     Result: PASS
  -> Pass 3/3 for bfab0d3c...
     Result: PASS

[3/80] Testing Logic Family: 7edbbaf4...
  -> Pass 1/3 for 7edbbaf4...
     Result: PASS
  -> Pass 2/3 for 7edbbaf4...
     Result: PASS
  -> Pass 3/3 for 7edbbaf4...
     Result: PASS

[4/80] Testing Logic Family: 7edbbaf4...
  -> Pass 1/3 for 7edbbaf4...
     Result: PASS
  -> Pass 2/3 for 7edbbaf4...
     Result: PASS
  -> Pass 3/3 for 7edbbaf4...
     Result: PASS

[5/80] Testing Logic Family: 9992f637...
  -> Pass 1/3 for 9992f637...
     Result: PASS
  -> Pass 2/3 for 9992f637...
     Result: PASS
  -> Pass 3/3 for 9992f637...
     Result: PASS

[6/80] Testing Logic Family: 9992f637...
  -> Pass 1/3 for 9992f637...
     Result: PASS
  -> Pass 2/3 for 9992f637...
     Result: PASS
  ->

Dataframe Visualization - but JSON should be viewable directly through Colab.

In [ ]:
# import pandas as pd
# df = pd.read_json("benchmark_results.json")
# review_queue = df[df['requires_manual_review'] == True]
# print(review_queue)

In [ ]:
# The In-Memory Visual Renderer
# Next, we swap out the circuit drawing logic for the schemdraw K-Map logic. This creates the image and passes it directly to your existing perturbation module.

def generate_kmap_image(circuit_data):
    """
    Renders a clean 4-variable Karnaugh Map from the provided minterms.
    Returns an in-memory PIL object.
    """
    minterms = circuit_data["minterms"]

    # Schemdraw expects a truth table formatted as a list of binary string tuples
    # e.g., [('0000', '1'), ('0001', '0'), ...]
    truthtable = []
    for i in range(16):
        # Convert decimal index to 4-bit binary string (e.g., 5 -> '0101')
        b_str = format(i, '04b')
        val = '1' if i in minterms else '0'
        truthtable.append((b_str, val))

    with schemdraw.Drawing(show=False) as d:
        d.config(fontsize=16)

        # Draw the K-Map natively.
        # names='ABCD' maps to the axes: AB on the side, CD on the top.
        d.add(logic.Kmap(names='ABCD', truthtable=truthtable))

        # Capture directly to memory
        image_bytes = d.get_imagedata('png')
        img_buffer = io.BytesIO(image_bytes)

    return Image.open(img_buffer)

In [ ]:
#The Karnaugh Map Generator
#This module defines the pedagogical templates, mixes them with a requested number of random maps, solves them, and packages them for the evaluation loop.

import io
import random
import hashlib
from sympy import symbols
from sympy.logic.boolalg import SOPform
import schemdraw
import schemdraw.logic as logic
from PIL import Image

def generate_kmap_dataset(num_random=20):
    """
    Generates a dataset of 4-variable K-Maps containing both
    targeted pedagogical templates and randomized distributions.
    """
    A, B, C, D = symbols('A B C D')
    dataset = []

    # --- 1. Pedagogical Templates ---
    # We define these by their 'minterms' (the decimal indices 0-15 where the output is 1)
    templates = {
        "Four Corners": [0, 2, 8, 10],            # Tests 4-corner wrap-around spatial reasoning
        "Top & Bottom Edges": [0, 1, 4, 5, 8, 9, 12, 13], # Tests vertical cylinder wrap-around
        "Overlapping Quads": [5, 7, 13, 15, 14, 10],      # Tests shared cell allocation
        "Checkerboard (XOR)": [1, 2, 4, 7, 8, 11, 13, 14],# No groupings possible, tests isolation
        "Central Block": [5, 7, 13, 15],          # Simple standard grouping
        "Single Column": [3, 2, 11, 10]           # Tests vertical alignment crossing gray-code axis
    }

    for name, minterms in templates.items():
        dataset.append({
            "type": "Template",
            "name": name,
            "minterms": minterms
        })

    # --- 2. Randomized Distributions ---
    random.seed(42) # Reproducible randomness
    for i in range(num_random):
        # Pick a random number of 1s to place (between 3 and 12 for non-trivial maps)
        num_ones = random.randint(3, 12)
        random_minterms = sorted(random.sample(range(16), num_ones))

        dataset.append({
            "type": "Random",
            "name": f"Random_{i+1}",
            "minterms": random_minterms
        })

    # --- 3. Ground Truth Resolution ---
    # Calculate the exact boolean minimum for every map in the dataset
    for data in dataset:
        # SOPform uses the Quine-McCluskey algorithm to find the Sum-of-Products minimum
        ground_truth_expr = SOPform([A, B, C, D], data["minterms"])

        # Create a unique logical hash
        family_id = hashlib.md5(str(ground_truth_expr).encode()).hexdigest()[:8]

        data["family_id"] = family_id
        data["expression"] = str(ground_truth_expr)

    return dataset

In [ ]:
import os
import re
import time
import google.generativeai as genai
from sympy import sympify
from sympy.logic.boolalg import Equivalent

genai.configure(api_key="YOUR_KEY_HERE")  # Replace with your actual API key


def evaluate_with_gemini(model ,pil_image, ground_truth_expr):
    """
    Sends the perturbed image to Gemini, extracts the logical answer,
    and mathematically verifies it against the ground truth.
    """


    prompt = """
    You are an expert computer engineering professor. 
    Analyze the Karnaugh Map (K-Map) in the provided image. 
    Write out the mathematically minimized Sum-of-Products (SOP) boolean expression for this map.
    
    You MUST provide your final answer on a new line in this exact format:
    FINAL ANSWER: <expression>
    
    CRITICAL SYNTAX RULES for the final expression:
    1. Use uppercase letters for variables (A, B, C, D).
    2. You MUST use the tilde (~) for NOT. Do NOT use apostrophes or primes (A').
    3. You MUST use the ampersand (&) for AND. Do NOT write variables next to each other (AB).
    4. You MUST use the pipe (|) for OR. Do NOT use the plus sign (+).
    5. Do NOT use the caret symbol (^) for XOR. Expand XOR into standard SOP form.
    
    EXAMPLE OF BAD OUTPUT: A'B + CD'
    EXAMPLE OF GOOD OUTPUT: ~A & B | C & ~D
    """

    try:
        # 4. The API Call (Passing the PIL image directly!)
        response = model.generate_content([prompt, pil_image])
        model_text = response.text

        # 5. Extract the answer using Regex
        match = re.search(r"FINAL ANSWER:\s*(.+)", model_text, re.IGNORECASE)

        if not match:
            return {
                "success": False,
                "error": "Formatting Failure: Model did not use 'FINAL ANSWER:' tag.",
                "raw_output": model_text
            }


        model_expr_str = match.group(1).strip()
        
        # --- 1. THE STRING SAFETY NET ---
        # If the model uses a plus sign for OR, fix it
        model_expr_str = model_expr_str.replace("+", "|")
        # If the model uses apostrophes for NOT (A'), swap to ~A
        model_expr_str = re.sub(r'([a-zA-Z])\'', r'~\1', model_expr_str)
        # If the model concatenates variables (AB) or (A~B), force an explicit AND (&)
        model_expr_str = re.sub(r'([A-Z])\s*(?=[A-Z~])', r'\1 & ', model_expr_str)
        # --------------------------------

        try:
            from sympy.logic.boolalg import simplify_logic, Equivalent
            
            # Parse the aggressively cleaned string
            model_sym = parse_expr(model_expr_str, transformations=LOGIC_TRANSFORMATIONS)
            
            # --- 2. THE STRICT EQUIVALENCE CHECK ---
            # simplify_logic forces SymPy to mathematically resolve the equivalence.
            # If they are equal, it returns a literal True. 
            # If not, it returns a boolean expression, which evaluates to False in this strict check.
            is_correct = (simplify_logic(Equivalent(ground_truth_expr, model_sym)) == True)
            
            return {
                "success": True,
                "is_correct": is_correct,
                "model_expression": model_expr_str,
                "ground_truth": str(ground_truth_expr),
                "raw_output": model_text
            }

        except Exception as math_err:
            return {
                "success": False,
                "error": f"Parse Error: Could not convert model output to SymPy logic. ({math_err})",
                "raw_output": model_text
            }

    except Exception as api_err:
        return {
            "success": False,
            "error": f"API Error: {api_err}"
        }

In [ ]:
import os
import re
from openai import OpenAI
from sympy.parsing.sympy_parser import parse_expr
from sympy.logic.boolalg import Equivalent

# Initialize the OpenAI client
openai_client = OpenAI(api_key="YOUR_KEY_HERE")  # Replace with your actual API key

def evaluate_with_openai(pil_image, ground_truth_sym):
    base64_image = pil_to_base64(pil_image)

    prompt = """
    You are an expert computer engineering professor. 
    Analyze the Karnaugh Map (K-Map) in the provided image. 
    Write out the mathematically minimized Sum-of-Products (SOP) boolean expression for this map.
    
    You MUST provide your final answer on a new line in this exact format:
    FINAL ANSWER: <expression>
    
    CRITICAL SYNTAX RULES for the final expression:
    1. Use uppercase letters for variables (A, B, C, D).
    2. You MUST use the tilde (~) for NOT. Do NOT use apostrophes or primes (A').
    3. You MUST use the ampersand (&) for AND. Do NOT write variables next to each other (AB).
    4. You MUST use the pipe (|) for OR. Do NOT use the plus sign (+).
    5. Do NOT use the caret symbol (^) for XOR. Expand XOR into standard SOP form.
    
    EXAMPLE OF BAD OUTPUT: A'B + CD'
    EXAMPLE OF GOOD OUTPUT: ~A & B | C & ~D
    """

    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0.0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ]
        )

        model_text = response.choices[0].message.content
        match = re.search(r"FINAL ANSWER:\s*(.+)", model_text, re.IGNORECASE)

        if not match:
            return {"success": False, "error": "Formatting Failure: No 'FINAL ANSWER:' tag.", "raw_output": model_text}

        model_expr_str = match.group(1).strip()

        model_expr_str = match.group(1).strip()
        
        # --- 1. THE STRING SAFETY NET ---
        # If the model uses a plus sign for OR, fix it
        model_expr_str = model_expr_str.replace("+", "|")
        # If the model uses apostrophes for NOT (A'), swap to ~A
        model_expr_str = re.sub(r'([a-zA-Z])\'', r'~\1', model_expr_str)
        # If the model concatenates variables (AB) or (A~B), force AND (&)
        model_expr_str = re.sub(r'([A-Z])\s*(?=[A-Z~])', r'\1 & ', model_expr_str)
        # --------------------------------

        try:
            from sympy.logic.boolalg import simplify_logic, Equivalent
            
            # Parse the cleaned string
            model_sym = parse_expr(model_expr_str, transformations=LOGIC_TRANSFORMATIONS)
            
            # --- 2. THE EQUIVALENCE CHECK ---
            # simplify_logic forces SymPy to mathematically resolve the equivalence.
            # If they are equal, it returns True. 
            # If not, it returns a boolean expression, which evaluates to False.
            is_correct = (simplify_logic(Equivalent(ground_truth_sym, model_sym)) == True)
            
            return {
                "success": True,
                "is_correct": is_correct,
                "model_expression": model_expr_str,
                "ground_truth": str(ground_truth_sym),
                "raw_output": model_text
            }
        except Exception as math_err:
            return {"success": False, "error": f"Parse Error: {math_err}", "raw_output": model_text}

    except Exception as api_err:
        return {"success": False, "error": f"API Error: {api_err}"}

In [ ]:
import os
import re
from anthropic import Anthropic
from sympy.parsing.sympy_parser import parse_expr
from sympy.logic.boolalg import Equivalent

# Initialize the Anthropic client
anthropic_client = Anthropic(api_key="YOUR_KEY_HERE")  # Replace with your actual API key

def evaluate_with_anthropic(pil_image, ground_truth_sym):
    base64_image = pil_to_base64(pil_image)

    prompt = """
    You are an expert computer engineering professor. 
    Analyze the Karnaugh Map (K-Map) in the provided image. 
    Write out the mathematically minimized Sum-of-Products (SOP) boolean expression for this map.
    
    You MUST provide your final answer on a new line in this exact format:
    FINAL ANSWER: <expression>
    
    CRITICAL SYNTAX RULES for the final expression:
    1. Use uppercase letters for variables (A, B, C, D).
    2. You MUST use the tilde (~) for NOT. Do NOT use apostrophes or primes (A').
    3. You MUST use the ampersand (&) for AND. Do NOT write variables next to each other (AB).
    4. You MUST use the pipe (|) for OR. Do NOT use the plus sign (+).
    5. Do NOT use the caret symbol (^) for XOR. Expand XOR into standard SOP form.
    
    EXAMPLE OF BAD OUTPUT: A'B + CD'
    EXAMPLE OF GOOD OUTPUT: ~A & B | C & ~D
    """

    try:
        response = anthropic_client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=1024,
            temperature=0.0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/png",
                                "data": base64_image
                            }
                        },
                        {"type": "text", "text": prompt}
                    ]
                }
            ]
        )

        model_text = response.content[0].text
        match = re.search(r"FINAL ANSWER:\s*(.+)", model_text, re.IGNORECASE)

        if not match:
            return {"success": False, "error": "Formatting Failure: No 'FINAL ANSWER:' tag.", "raw_output": model_text}

        model_expr_str = match.group(1).strip()

        model_expr_str = match.group(1).strip()
        
        # --- 1. THE STRING SAFETY NET ---
        # If the model uses a plus sign for OR, fix it
        model_expr_str = model_expr_str.replace("+", "|")
        # If the model uses apostrophes for NOT (A'), swap to ~A
        model_expr_str = re.sub(r'([a-zA-Z])\'', r'~\1', model_expr_str)
        # If the model concatenates variables (AB) or (A~B), force an explicit AND (&)
        model_expr_str = re.sub(r'([A-Z])\s*(?=[A-Z~])', r'\1 & ', model_expr_str)
        # --------------------------------

        try:
            from sympy.logic.boolalg import simplify_logic, Equivalent
            
            # Parse the aggressively cleaned string
            model_sym = parse_expr(model_expr_str, transformations=LOGIC_TRANSFORMATIONS)
            
            # --- 2. THE STRICT EQUIVALENCE CHECK ---
            # simplify_logic forces SymPy to mathematically resolve the equivalence.
            # If they are equal, it returns a literal True. 
            # If not, it returns a boolean expression, which evaluates to False in this strict check.
            is_correct = (simplify_logic(Equivalent(ground_truth_sym, model_sym)) == True)
            
            return {
                "success": True,
                "is_correct": is_correct,
                "model_expression": model_expr_str,
                "ground_truth": str(ground_truth_sym),
                "raw_output": model_text
            }
        except Exception as math_err:
            return {"success": False, "error": f"Parse Error: {math_err}", "raw_output": model_text}

    except Exception as api_err:
        return {"success": False, "error": f"API Error: {api_err}"}

K-map Generation Code and Model Evaluation Loop

In [ ]:
import io
import json
import time
import datetime
import os
from pathlib import Path
import schemdraw
import schemdraw.logic as logic
from sympy import sympify
from PIL import Image

from sympy.parsing.sympy_parser import parse_expr, standard_transformations, convert_xor

# Define an environment that prevents ^ from becoming Power (**)
LOGIC_TRANSFORMATIONS = tuple(t for t in standard_transformations if t != convert_xor)

def generate_kmap_image(circuit_data):
    """
    Renders a clean 4-variable Karnaugh Map from the provided minterms.
    Returns an in-memory PIL object.
    """
    minterms = circuit_data["minterms"]

    # Schemdraw expects a truth table formatted as a list of binary string tuples
    # e.g., [('0000', '1'), ('0001', '0'), ...]
    truthtable = []
    for i in range(16):
        # Convert decimal index to 4-bit binary string (e.g., 5 -> '0101')
        b_str = format(i, '04b')
        val = '1' if i in minterms else '0'
        truthtable.append((b_str, val))

    with schemdraw.Drawing(show=False) as d:
        d.config(fontsize=16)

        # Draw the K-Map natively.
        # names='ABCD' maps to the axes: AB on the side, CD on the top.
        d.add(logic.Kmap(names='ABCD', truthtable=truthtable))

        # Capture directly to memory
        image_bytes = d.get_imagedata('png')
        img_buffer = io.BytesIO(image_bytes)

    return Image.open(img_buffer)

# ---------------------------------------------------------
# 2. Main Evaluation Loop
# ---------------------------------------------------------
def run_automated_benchmark(dataset,
                            perturb_config,
                            experiment_name="run_01",
                            sample_size=5,
                            num_passes=3, 
                            save_images=False,
                            base_output_dir="benchmark_data"):

    # 1. Define the deterministic model settings
    # We use temperature=0.0 to minimize stochastic creativity in math problems
    generation_config = {
        "temperature": 0.0,
        "top_p": 1.0,
    }

    # 2. Initialize the Pro model
    shared_model = genai.GenerativeModel(
        model_name="gemini-2.5-pro",
        generation_config=generation_config
    )

    exp_dir = Path(base_output_dir) / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    output_file = exp_dir / "results.json"

    if save_images:
        clean_dir = exp_dir / "clean"
        perturb_dir = exp_dir / "perturbed"
        clean_dir.mkdir(exist_ok=True)
        perturb_dir.mkdir(exist_ok=True)

    if output_file.exists():
        with open(output_file, 'r') as f:
            results_log = json.load(f)
    else:
        results_log = []
        with open(output_file, 'w') as f:
            json.dump([], f)

    print(f"Starting benchmark '{experiment_name}'. {num_passes} passes per image.")

    for i, data in enumerate(dataset[:sample_size]):
        family_id = data['family_id']
        print(f"\n[{i+1}/{sample_size}] Testing Logic Family: {family_id}...")

        try:
            # --- 1. GENERATE ONCE ---
            clean_pil = generate_kmap_image(data)

            # --- 2. PERTURB ONCE  ---
            adversarial_pil = apply_perturbations(clean_pil, **perturb_config)

            if not adversarial_pil:
                raise ValueError("Perturbation function returned None.")

            if save_images:
                clean_pil.save(clean_dir / f"{family_id}_clean.png")
                adversarial_pil.save(perturb_dir / f"{family_id}_perturbed.png")

            ground_truth_sym = parse_expr(data['expression'], transformations=LOGIC_TRANSFORMATIONS)
            #ground_truth_sym = parse_expr(data['expression'], transformations=standard_transformations)
            print(ground_truth_sym)

            # --- 3. EVALUATE N TIMES WITH SMART RETRY ---
            for pass_idx in range(1, num_passes + 1):
                print(f"  -> Pass {pass_idx}/{num_passes} for {family_id}...")

                max_retries = 3
                retry_count = 0
                evaluation = None

                while retry_count < max_retries:
                    # We pass the EXACT SAME image to the API
                    evaluation = evaluate_with_gemini(shared_model, adversarial_pil, ground_truth_sym)
                    #evaluation = evaluate_with_openai(adversarial_pil, ground_truth_sym)
                    #evaluation = evaluate_with_anthropic(adversarial_pil, ground_truth_sym)


                    error_text = str(evaluation.get("error", ""))
                    if not evaluation.get("success") and ("429" in error_text or "timed out" in error_text.lower()):

                        if "429" in error_text:
                            match = re.search(r"Please retry in ([0-9.]+)s", error_text)
                            wait_time = float(match.group(1)) + 5.0 if match else 60.0
                            print(f"     [Rate Limit] Sleeping for {wait_time:.1f}s...")
                        else:
                            wait_time = 30.0  # Cool-down for proxy timeouts
                            print(f"     [Proxy Timeout] Broken socket detected. Sleeping {wait_time}s before retry...")

                        time.sleep(wait_time)
                        retry_count += 1
                        continue

                    break

                # If we exhausted retries and it's still a 429, mark it as a hard error
                if not evaluation.get("success") and "429" in str(evaluation.get("error", "")):
                     evaluation["error"] = "CRITICAL: Exhausted all rate limit retries."

                # --- LOGGING ---
                log_entry = {
                    "timestamp": datetime.datetime.now().isoformat(),
                    "experiment_name": experiment_name,
                    "family_id": family_id,
                    "pass_index": pass_idx,
                    "ground_truth_logic": str(ground_truth_sym),
                    "perturbation_config": perturb_config,
                    "api_success": evaluation.get("success", False),
                    "is_correct": evaluation.get("is_correct", None),
                    "model_parsed_logic": evaluation.get("model_expression", None),
                    "raw_model_output": evaluation.get("raw_output", ""),
                    "error_message": evaluation.get("error", None),
                    "requires_manual_review": not evaluation.get("success", False)
                }

                results_log.append(log_entry)

                with open(output_file, 'w') as f:
                    json.dump(results_log, f, indent=4)

                if log_entry["api_success"]:
                    status = "PASS" if log_entry["is_correct"] else "FAIL"
                    print(f"     Result: {status}")
                else:
                    print(f"     Result: FLAG FOR REVIEW ({log_entry['error_message']})")

                # Standard sleep between successful passes
                time.sleep(15)

        except Exception as e:
            print(f"  -> CRITICAL EXCEPTION on {family_id}: {e}")
            log_entry = {
                "timestamp": datetime.datetime.now().isoformat(),
                "experiment_name": experiment_name,
                "family_id": family_id,
                "pass_index": "N/A",
                "error_message": f"SYSTEM EXCEPTION: {str(e)}",
                "requires_manual_review": True
            }
            results_log.append(log_entry)
            with open(output_file, 'w') as f:
                json.dump(results_log, f, indent=4)
            time.sleep(4)

    print(f"\nBenchmark complete. Data safely stored in {exp_dir}")
    return results_log

if __name__ == "__main__":
    # 1. Generate the data and explicitly request the variants
    dataset = generate_kmap_dataset(num_random=24)



    # 3. Full Perturbation Suite
    my_perturb_config = {
        "photocopy": True,
        "photocopy_cycles": 3,
        "noise": True,
        "affine": True,
        "lines": True
    }

    clean_baseline_config = {
        "photocopy": False,
        "noise": False,
        "affine": False,
        "lines": False
    }

    # 4. Execute the Benchmark
    # Note: sample_size is set to the full length of the paired list
    run_automated_benchmark(
        dataset=dataset,

        #Perturbations:
        #perturb_config=my_perturb_config,
        perturb_config=clean_baseline_config,

        #Experiments:
        experiment_name="kmap-geminipro25_clean",
        #experiment_name="kmap-gpt40mini_perturbed",
        #experiment_name="kmap-claudehaiku45_perturbed",

        sample_size=len(dataset),
        num_passes=3,
        save_images=True
    )

Starting benchmark 'kmap-geminipro25_clean'. 3 passes per image.

[1/10] Testing Logic Family: 259270d3...
~B & ~D
  -> Pass 1/3 for 259270d3...


/tmp/ipykernel_662055/655588362.py:116: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PiecewiseAffineTransform.from_estimate` class constructor instead.
  tform.estimate(src, dst)


     Result: PASS
  -> Pass 2/3 for 259270d3...
     Result: PASS
  -> Pass 3/3 for 259270d3...
     Result: PASS

[2/10] Testing Logic Family: 6f455cb6...
~C
  -> Pass 1/3 for 6f455cb6...
     Result: PASS
  -> Pass 2/3 for 6f455cb6...
     Result: PASS
  -> Pass 3/3 for 6f455cb6...
     Result: PASS

[3/10] Testing Logic Family: 7a5f8ab0...
(B & D) | (A & C & ~D)
  -> Pass 1/3 for 7a5f8ab0...
     Result: PASS
  -> Pass 2/3 for 7a5f8ab0...
     Result: PASS
  -> Pass 3/3 for 7a5f8ab0...
     Result: PASS

[4/10] Testing Logic Family: 706b2b1c...
(A & B & C & ~D) | (A & B & D & ~C) | (A & C & D & ~B) | (B & C & D & ~A) | (A & ~B & ~C & ~D) | (B & ~A & ~C & ~D) | (C & ~A & ~B & ~D) | (D & ~A & ~B & ~C)
  -> Pass 1/3 for 706b2b1c...
     Result: PASS
  -> Pass 2/3 for 706b2b1c...
     Result: PASS
  -> Pass 3/3 for 706b2b1c...
     Result: PASS

[5/10] Testing Logic Family: 190539a1...
B & D
  -> Pass 1/3 for 190539a1...
     Result: PASS
  -> Pass 2/3 for 190539a1...
     Result: PASS


In [ ]:
import json
import pandas as pd
from pathlib import Path

def analyze_benchmark_logs(json_path):
    """
    Parses a benchmark JSON log and calculates  Pass@K metrics.
    """
    path = Path(json_path)
    if not path.exists():
        raise FileNotFoundError(f"Could not find {json_path}")
        
    with open(path, 'r') as f:
        data = json.load(f)

    # 1. Load into Pandas
    df = pd.DataFrame(data)

    # 2. Clean the Data
    # If the API failed entirely, 'is_correct' might be None/NaN. Force to False.
    df['is_correct'] = df['is_correct'].fillna(False).astype(bool)

    # 3. Aggregate by Problem (family_id)
    # We want to know how many times out of the N passes it got this specific problem right.
    group_cols = ['family_id']
    
    # If we are parsing the Circuit dataset, keep the variant tags for paired analysis
    if 'is_variant' in df.columns:
        group_cols.extend(['is_variant', 'variant_note'])

    results = df.groupby(group_cols).agg(
        total_passes=('pass_index', 'count'),
        successful_passes=('is_correct', 'sum')
    ).reset_index()

    # 4. Calculate the Rigor Tiers
    results['pass_any'] = results['successful_passes'] >= 1
    results['majority_vote'] = results['successful_passes'] >= 2
    results['strict_pass'] = results['successful_passes'] == 3

    # 5. Generate the Summary Statistics
    total_problems = len(results)
    summary = {
        "Total Unique Problems": total_problems,
        "Pass@Any (>=1 Pass)": (results['pass_any'].sum() / total_problems) * 100,
        "Majority Vote (>=2 Passes)": (results['majority_vote'].sum() / total_problems) * 100,
        "Strict Consistency (3 Passes)": (results['strict_pass'].sum() / total_problems) * 100
    }

    return results, summary

# ==========================================
# Execution
# ==========================================
if __name__ == "__main__":
    # Example: Parsing the Gemini Clean Baseline

    logs = [ 

        # "ld_claudeHaiku45_clean/results.json",
        # "ld_claudeHaiku45_perturbed/results.json",
        # "ld_gpt4omini_clean/results.json",
        # "ld_gpt4omini_perturbed/results.json",
        #"ld_gemini25pro_clean/results.json",
        #"ld_gemini25pro_perturbed/results.json"

        # "benchmark_data/kmap-claudehaiku45_clean/results.json",
        # "benchmark_data/kmap-claudehaiku45_perturbed/results.json",
        # "benchmark_data/kmap-gpt40mini_clean/results.json",
        # "benchmark_data/kmap-gpt40mini_perturbed/results.json",
        "benchmark_data/kmap-geminipro25_clean/results.json",
        "benchmark_data/kmap-geminipro25_perturbed/results.json"
    ]

    for log_file in logs:
    
        try:
            detailed_df, summary_stats = analyze_benchmark_logs(log_file)
            
            print(f"--- Analysis for: {log_file} ---")
            for metric, value in summary_stats.items():
                if "Total" in metric:
                    print(f"{metric}: {value}")
                else:
                    print(f"{metric}: {value:.1f}%")
                    
        except FileNotFoundError as e:
            print(e)

--- Analysis for: /home/vm_burger/Downloads/benchmark_data/kmap-geminipro25_clean/results.json ---
Total Unique Problems: 10
Pass@Any (>=1 Pass): 100.0%
Majority Vote (>=2 Passes): 90.0%
Strict Consistency (3 Passes): 90.0%
--- Analysis for: /home/vm_burger/Downloads/benchmark_data/kmap-geminipro25_perturbed/results.json ---
Total Unique Problems: 10
Pass@Any (>=1 Pass): 100.0%
Majority Vote (>=2 Passes): 100.0%
Strict Consistency (3 Passes): 70.0%
